# 02 · Census (TIGER block groups + ACS 5-year) and GTFS stops

Same code as `make census` / `make gtfs`. Block-group ACS estimates have large margins of error:
CVs (MOE / 1.645 / estimate) above 30 % are flagged and must be caveated wherever shown.

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
cfg.ensure_dirs()
print(
    cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year, "| EPSG", cfg.analysis_epsg
)
print("raw:", cfg.raw_dir)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from urban_access import census, gtfs

counties = census.detect_counties(cfg)
print("counties (state, county):", counties)
assert ("32", "510") in counties  # Carson City; the bbox also reaches Douglas County (005)
census.run(cfg)
bg = gpd.read_parquet(cfg.raw_dir / "block_groups.parquet")
acs = pd.read_parquet(cfg.raw_dir / "acs.parquet")
print(len(bg), "block groups;", int(bg["scored"].sum()), "scored (centroid in bbox)")
bg[bg["scored"]].groupby("countyfp").size()

In [ ]:
scored = bg[bg["scored"]].merge(acs, on="geoid", how="left", validate="1:1")
assert scored["pop_total"].notna().all(), "scored block group without ACS"
assert scored["tract_geoid"].str.len().eq(11).all()
print(
    "ACS year", cfg.acs_year, "| population in scored block groups:", int(scored["pop_total"].sum())
)
print("zero-population block groups:", scored.loc[scored["pop_total"] == 0, "geoid"].tolist())
fields = ["pop_total", "median_hh_income", "zero_car_hh_pct", "pop_65plus_pct", "pop_under18_pct"]
scored[fields].describe().round(1)

In [ ]:
cv_cols = ["pop_total_cv", "median_hh_income_cv", "zero_car_hh_pct_cv"]
print("share of scored block groups with CV > 30 %:")
print((scored[cv_cols] > 30).mean().round(2))
scored.sort_values("median_hh_income_cv", ascending=False)[
    [
        "geoid",
        "pop_total",
        "median_hh_income",
        "median_hh_income_moe",
        "median_hh_income_cv",
        "zero_car_hh_pct",
        "zero_car_hh_pct_moe",
    ]
].head(10).round(1)

In [ ]:
stops_path = gtfs.run(cfg)
stops = gpd.read_parquet(stops_path) if stops_path else None
if stops is not None:
    assert len(stops) > 0 and stops.geometry.notna().all()
    in_scored = stops.sjoin(scored[["geoid", "geometry"]], predicate="within")
    print(len(stops), "served stops;", len(in_scored), "inside scored block groups")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, col, title in zip(
    axes,
    ["pop_total", "median_hh_income", "zero_car_hh_pct"],
    ["Population", "Median household income ($)", "Households with no car (%)"],
    strict=True,
):
    bg[~bg["scored"]].plot(ax=ax, color="#eee", edgecolor="white")
    scored.plot(
        ax=ax,
        column=col,
        legend=True,
        cmap="viridis",
        edgecolor="white",
        linewidth=0.3,
        missing_kwds={"color": "lightgray", "hatch": "//"},
    )
    if stops is not None:
        stops.plot(ax=ax, color="red", markersize=2)
    ax.set_title(f"{title} — ACS {cfg.acs_year} 5-yr")
    ax.set_xlim(*scored.total_bounds[[0, 2]])
    ax.set_ylim(*scored.total_bounds[[1, 3]])
    ax.set_axis_off()
plt.tight_layout()